In [1]:
import pandas as pd
import scipy

from eval_functions import correctness_score, hallucination_score, hallucination_score_noramlized

#### Scoring criteria: Correctness

- Score 0 — The generated answer contains incorrect or potentially harmful advice that contradicts the reference answer. This is the worst outcome.
- Score 1 — The generated answer is incomplete or omits important details from the reference, but contains nothing incorrect or harmful.
- Score 2 — The generated answer is correct and captures the key facts from the reference answer.

#### Scoring criteria: Hallucination

- Score 0 — The answer contains coaching advice that directly contradicts the retrieved context, or invents training recommendations with no basis in the context. This is the worst outcome.
- Score 1 — The coaching advice is mostly supported by the context but includes minor unsupported recommendations.
- Score 2 — All coaching advice and recommendations are directly supported by the retrieved context. Visual observations about the user's form do not count against this score.

## T-test hypotheses 

- H0: prompt or model change did not have any statistically significant impact on results
- H1: prompt or model change did have a statistically significant impact on results 

# Test 31 (baseline) 

#### Vision-faithfulness-rag-updated-system-prompt-test3

- updated system prompt with new behavior instructions 

        # BEHAVIOR INSTRUCTIONS
        1. Tone
        - You're eager and excited to help 
        2. How to analyze
        - Break down your analysis into three sections
            What looks good
                - 1 to 2 points
            Main fixes
                - Cover all significant issues you observe
            Mental cues
                - A brief list of mental cues the lifter can easily remember during their lift

In [2]:
test_31_baseline = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-rag-updated-system-prompt-test3.csv")

print("Test 31: Hallucination")
test_31_hallucination = hallucination_score_noramlized(test_31_baseline, df_name="test_31")
hallucination_score_mean = test_31_baseline["hallucination"].mean() * 2 # denormalize the overall score
print(f"Baseline hallucination score: {hallucination_score_mean}")

print(" ")

print("Test 31: Correctness")
test_31_correctness = correctness_score(test_31_baseline, df_name="test_31")
correctness_score_mean = test_31_baseline["correctness"].mean() * 2 # denormalize the overall score
print(f"Baseline correctness score: {correctness_score_mean}")

Test 31: Hallucination
95% CI Hallucination Low - test_31: 1.35
95% CI Hallucination High - test_31: 1.8
Standard Error - Hallucination - test_31: 0.1101
hallucination_MOE - test_31: 0.22
Baseline hallucination score: 1.6
 
Test 31: Correctness
95% CI Correctness Low - test_31: 1.3
95% CI Correctness High - test_31: 1.7
Standard Error - Correctness - test_31: 0.1137
Correctness MOE - test_31: 0.20
Baseline correctness score: 1.5


# Test 33 
- Model Swap Comparison: gpt-5 → gpt-5.4 (Response Generator)

In [3]:
test_33 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-gpt-5.4-response_generator.csv")
test_33.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,20.0,0.0,20.000000,20.000000,20.000000,20.000000,20.000000
mean,1.0,NaN,24.849006,14675.300000,0.041166,0.725000,0.750000
std,0.0,NaN,4.092166,6002.343271,0.014363,0.255209,0.256495
min,1.0,NaN,20.083819,6416.000000,0.020617,0.500000,0.500000
25%,1.0,NaN,22.820954,9829.750000,0.029325,0.500000,0.500000
50%,1.0,NaN,24.168265,11795.000000,0.033617,0.500000,0.750000
75%,1.0,NaN,25.836488,21842.500000,0.057639,1.000000,1.000000
max,1.0,NaN,37.365576,22000.000000,0.059112,1.000000,1.000000


In [4]:
print("Test 33: Hallucination")
vision_20_hallucination_test_4 = hallucination_score_noramlized(test_33, df_name="test_33")
hallucination_score_mean = test_33["hallucination"].mean() * 2 # denormalize the overall score
print(f"Test 33 Hallcination score: {hallucination_score_mean}")

print(" ")

print("Test 33: Correctnes")
vision_20_correctness_test_4 = correctness_score(test_33, df_name="test_33")
correctness_score_mean = test_33["correctness"].mean() * 2 # denormalize the overall score
print(f"Test 33 correctness score: {correctness_score_mean}")

Test 33: Hallucination
95% CI Hallucination Low - test_33: 1.3
95% CI Hallucination High - test_33: 1.7
Standard Error - Hallucination - test_33: 0.1130
hallucination_MOE - test_33: 0.20
Test 33 Hallcination score: 1.5
 
Test 33: Correctnes
95% CI Correctness Low - test_33: 1.25
95% CI Correctness High - test_33: 1.65
Standard Error - Correctness - test_33: 0.1115
Correctness MOE - test_33: 0.20
Test 33 correctness score: 1.45


## T-test: baseline vs test 33

#### Correctness

In [5]:
baseline_scores = test_31_baseline["correctness"]
new_scores = test_33["correctness"]

correctness_result = scipy.stats.ttest_rel(baseline_scores, new_scores)
correctness_result

TtestResult(statistic=np.float64(0.36971698295049193), pvalue=np.float64(0.7156821076942621), df=np.int64(19))

#### Hallucination

In [6]:
baseline_scores = test_31_baseline["hallucination"]
new_scores = test_33["hallucination"]

hallucination_result = scipy.stats.ttest_rel(baseline_scores, new_scores)
hallucination_result


TtestResult(statistic=np.float64(0.6979824404521129), pvalue=np.float64(0.4936425876371031), df=np.int64(19))

## Results: 
#### Baseline model (gpt-5)
- Baseline hallucination score: 1.6
- Test 33 Hallcination score: 1.5
- Total cost: $0.83
- P50 Latency: 53.71s
- P99 Latency: 101.13s


#### Test 33 (gpt-5.4)
- Baseline correctness score: 1.5
- Test 33 correctness score: 1.45
- Total cost: $0.83
- P50 Latency: 24.17s (29s faster)
- P99 Latency: 35.39s (51s faster)

### Recommendations

Change in scores is not statistically significant (p > 0.05). Major reduction in latency for zero cost increase. Ship model. 

# Test 34 
- System prompt update: 100 word output cap

Updated:

            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 
    Inspect each image CLOSELY and carefully. Look for issues realted to form, safety, and unhelpful camera angles.
    Be specific about what you observe and include that in your feedback.
             
        # RESPONSE STYLE
    - Lead with your most important observations in 2-3 sentences.
    - Give ONE specific, actionable cue the user can try on their next set.
    - If you notice additional areas to improve, briefly mention them so the user can ask follow-up questions.
    - Keep your total response under 100 words unless the user asks for more detail.
    - When the user asks a follow-up, answer ONLY that question. Do not repeat your full analysis.
    - If the lift looks solid overall, say so. Do not nitpick to fill space.

    # ANSWER CONTEXT
    First describe what you observe in the images.         
    Then use the ONLY following context to provide coaching advice:
 
    {top_k_chunks}
    
    If the query or image isn't in context, reply, 'I don't have expert coaching advice for this exercise yet. 
    Currently I can analyze: bench press, overhead press, incline bench press...'"
        

In [7]:
test_34 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-system-prompt-100-word-cap.csv")
test_34.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,20.0,0.0,20.000000,20.000000,20.000000,20.000000,20.000000
mean,1.0,NaN,9.390251,13468.300000,0.032615,0.500000,0.900000
std,0.0,NaN,1.504880,5982.938436,0.014094,0.229416,0.205196
min,1.0,NaN,7.044194,5563.000000,0.014102,0.000000,0.500000
25%,1.0,NaN,8.414615,8642.750000,0.021317,0.500000,1.000000
50%,1.0,NaN,9.372272,10466.000000,0.025466,0.500000,1.000000
75%,1.0,NaN,10.052803,20641.000000,0.049415,0.500000,1.000000
max,1.0,NaN,13.059238,20804.000000,0.050002,1.000000,1.000000


In [8]:
print("Test 34: Hallucination")
test_34_hallucination_test = hallucination_score_noramlized(test_34, df_name="test_34")
hallucination_score_mean = test_34["hallucination"].mean() * 2 # denormalize the overall score
print(f"Test 34 Hallcination score: {hallucination_score_mean}")

print(" ")

print("Test 34: Correctnes")
test_34_correctness_test = correctness_score(test_34, df_name="system_prompt_eval_v1")
correctness_score_mean = test_34["correctness"].mean() * 2 # denormalize the overall score
print(f"Test 34 Correctness score: {correctness_score_mean}")

Test 34: Hallucination
95% CI Hallucination Low - test_34: 1.55
95% CI Hallucination High - test_34: 1.95
Standard Error - Hallucination - test_34: 0.0891
hallucination_MOE - test_34: 0.20
Test 34 Hallcination score: 1.8
 
Test 34: Correctnes
95% CI Correctness Low - system_prompt_eval_v1: 0.8
95% CI Correctness High - system_prompt_eval_v1: 1.2
Standard Error - Correctness - system_prompt_eval_v1: 0.1008
Correctness MOE - system_prompt_eval_v1: 0.20
Test 34 Correctness score: 1.0


### Correctness

In [9]:
baseline_scores = test_33["correctness"]
new_scores = test_34["correctness"]

correctness_result = scipy.stats.ttest_rel(baseline_scores, new_scores)
correctness_result

TtestResult(statistic=np.float64(2.932194632545474), pvalue=np.float64(0.008551024036843772), df=np.int64(19))

#### Hallucination

In [10]:
baseline_scores = test_33["hallucination"]
new_scores = test_34["hallucination"]

hallucination_result = scipy.stats.ttest_rel(baseline_scores, new_scores)
hallucination_result

TtestResult(statistic=np.float64(-2.042236937115053), pvalue=np.float64(0.055256843138992665), df=np.int64(19))

## Results: 
#### Test 33 (new baseline)
- Test 33 correctness score: 1.45
- Test 33 Hallcination score: 1.5
- Total cost: $0.83
- P50 Latency: 24.17s
- P99 Latency: 35.39s

#### Test 34 (system prompt update)
- Test 34 Correctness score: 1.0 (.45 lower)
- Test 34 Hallcination score: 1.8
- Total cost: $0.66 ($0.18 lower)
- P50 Latency: 9.37s (14.8s faster)
- P99 Latency: 12.86s (23s faster)

### Recommendations

The concise prompt significantly impacts correctness (p < 0.05) but the hallucination increase is not quite significant (p = 0.055). However, this is a cheaper, faster system. More testing is needed.   

# Test 35
- 100 word cap system prompt + trimmed reference answers

Reference answers were trimmed from the "golden dataset" answers to include only the one or two most important details, since the hope is that the system can offer fast, cheap, concise answers that are also correct, instead of giving large, complex answers that a user is less likely to completely read in a gym setting. Ideally, the coach would offer 100 words on the most pressing one or two issues, and allow the user time to adjust, re-film, and try again, before moving on.

In [11]:
test_35 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-system-prompt-100-word-cap-concise-reference-answes.csv")
test_35.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,20.0,0.0,20.000000,20.000000,20.000000,20.000000,20.000000
mean,1.0,NaN,10.874275,13495.800000,0.032525,0.525000,0.775000
std,0.0,NaN,2.494822,5954.334923,0.014244,0.412789,0.255209
min,1.0,NaN,7.932170,5546.000000,0.013949,0.000000,0.500000
25%,1.0,NaN,8.543955,8533.750000,0.020406,0.000000,0.500000
50%,1.0,NaN,10.531850,10772.500000,0.026284,0.500000,1.000000
75%,1.0,NaN,12.651854,20570.000000,0.049363,1.000000,1.000000
max,1.0,NaN,17.050450,20762.000000,0.049872,1.000000,1.000000


In [12]:
# Test 35: 100 word cap system prompt + trimmed reference answers

print("Test 35: Hallucination")
test_35_hallucination_test = hallucination_score_noramlized(test_35, df_name="test_35")
hallucination_score_mean = test_35["hallucination"].mean() * 2 # denormalize the overall score
print(f"Test 35 Hallcination score: {hallucination_score_mean}")

print(" ")

print("Test 35: Correctnes")
test_35_correctness_test = correctness_score(test_35, df_name="test_35")
correctness_score_mean = test_35["correctness"].mean() * 2 # denormalize the overall score
print(f"Test 35 Correctness score: {correctness_score_mean}")

Test 35: Hallucination
95% CI Hallucination Low - test_35: 1.35
95% CI Hallucination High - test_35: 1.75
Standard Error - Hallucination - test_35: 0.1109
hallucination_MOE - test_35: 0.20
Test 35 Hallcination score: 1.55
 
Test 35: Correctnes
95% CI Correctness Low - test_35: 0.7
95% CI Correctness High - test_35: 1.4
Standard Error - Correctness - test_35: 0.1808
Correctness MOE - test_35: 0.35
Test 35 Correctness score: 1.05


#### Correctness t-test: Test 33 vs Test 35

In [13]:
baseline_scores = test_33["correctness"]
new_scores = test_35["correctness"]

correctness_result = scipy.stats.ttest_rel(baseline_scores, new_scores)
correctness_result

TtestResult(statistic=np.float64(2.026846838838127), pvalue=np.float64(0.056945350695465795), df=np.int64(19))

#### Hallucination t-test: Test 33 vs Test 35

In [14]:
baseline_scores = test_33["hallucination"]
new_scores = test_35["hallucination"]

hallucination_result = scipy.stats.ttest_rel(baseline_scores, new_scores)
hallucination_result

TtestResult(statistic=np.float64(-0.36971698295049193), pvalue=np.float64(0.7156821076942621), df=np.int64(19))

## Results: 
#### Test 33 (new baseline)
- Test 33 correctness score: 1.45
- Test 33 Hallcination score: 1.5
- Total cost: $0.83
- P50 Latency: 24.17s
- P99 Latency: 35.39s

#### Test 35 (system prompt update + trimmed reference answers)
- Test 35 Correctness score: 1.05 (-0.40)
- Test 35 Hallcination score: 1.55 (+0.05)
- Total cost: $0.66 (-$0.18)
- P50 Latency: 10.53s (-13.7s)
- P99 Latency: 16.51s (-19.4s)

### Recommendations

Hallucination score increase is not statistically significant (p > 0.05). Reduced correctness is not quite statistically significant (p = 0.056). However, it is outside the margin of error (0.35), and very close to Test 34's results correctness results (1.0), which were statistically significant. So this result is inconclusive, but likely significant. More testing is needed.    

# Test 36
- baseline system prompt with trimmed reference answers

In [15]:
test_36 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-baseline-and-trimmed-reference-answers.csv")
test_36.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,20.0,0.0,20.000000,20.000000,20.000000,20.000000,20.000000
mean,1.0,NaN,23.907613,14605.800000,0.040731,0.725000,0.725000
std,0.0,NaN,3.162018,5993.240074,0.014257,0.255209,0.255209
min,1.0,NaN,17.241544,6690.000000,0.021951,0.500000,0.500000
25%,1.0,NaN,21.194372,9584.750000,0.028635,0.500000,0.500000
50%,1.0,NaN,24.094976,11741.000000,0.034138,0.500000,0.500000
75%,1.0,NaN,26.002701,21811.000000,0.057273,1.000000,1.000000
max,1.0,NaN,29.817029,22283.000000,0.060704,1.000000,1.000000


In [16]:
# Test 36: baseline system prompt + trimmed reference answers

print("Test 36: Hallucination")
test_36_hallucination_test = hallucination_score_noramlized(test_36, df_name="test_36")
hallucination_score_mean = test_36["hallucination"].mean() * 2 # denormalize the overall score
print(f"Test 36 Hallucination score: {hallucination_score_mean}")

print(" ")

print("Test 36: Correctnes")
test_36_correctness_test = correctness_score(test_36, df_name="test_36")
correctness_score_mean = test_36["correctness"].mean() * 2 # denormalize the overall score
print(f"Test 36 Correctness score: {correctness_score_mean}")

Test 36: Hallucination
95% CI Hallucination Low - test_36: 1.25
95% CI Hallucination High - test_36: 1.65
Standard Error - Hallucination - test_36: 0.1127
hallucination_MOE - test_36: 0.20
Test 36 Hallucination score: 1.45
 
Test 36: Correctnes
95% CI Correctness Low - test_36: 1.25
95% CI Correctness High - test_36: 1.7
Standard Error - Correctness - test_36: 0.1124
Correctness MOE - test_36: 0.22
Test 36 Correctness score: 1.45


#### Correctness t-test: Test 33 (baseline) vs Test 36

In [17]:
baseline_scores = test_33["correctness"]
new_scores = test_36["correctness"]

correctness_result = scipy.stats.ttest_rel(baseline_scores, new_scores)
correctness_result

TtestResult(statistic=np.float64(0.0), pvalue=np.float64(1.0), df=np.int64(19))

#### Hallucination t-test: Test 33 (baseline) vs Test 36

In [18]:
baseline_scores = test_33["hallucination"]
new_scores = test_36["hallucination"]

correctness_result = scipy.stats.ttest_rel(baseline_scores, new_scores)
correctness_result

TtestResult(statistic=np.float64(0.3257994036161639), pvalue=np.float64(0.7481369754472712), df=np.int64(19))

## Results: 
#### Test 33 (new baseline)
- Test 33 correctness score: 1.45
- Test 33 Hallcination score: 1.5
- Total cost: $0.83
- P50 Latency: 24.17s
- P99 Latency: 35.39s

#### Test 36 (baseline system prompt update + trimmed reference answers)
- Test 36 Correctness score: 1.45
- Test 36 Hallcination score: 1.45 (-0.05)
- Total cost: $0.82 (-$0.1)
- P50 Latency: 24.09s
- P99 Latency: 29.63s

### Recommendations

Scoring the baseline prompt against trimmed reference answers returned identical correctness scores (1.45), confirming the trimmed references are a fair swap. There is a tradeoff between cost & latency, and accuracy. For now, accuracy remains more important, 30s is tolerable. But further testing is needed to find a balance between shorter answers and correct feedback.

# Test 37 
- Updated baseline system prompt 

        Do not make any mention to "frames". To the user, you are watching a video.
             
             
        # BEHAVIOR INSTRUCTIONS
        1. Tone
        - You're eager and excited to help 
        2. How to analyze
            Main fixes
                - Cover all significant issues you observe
            Wrap it up with a follow-up quesion
                - Offer one thing you can do next to help them time 
                    Eg, "If you want, I can also..."


        # ANSWER CONTEXT
        First describe what you observe in the images.         
        Then use the ONLY following context to provide coaching advice:

In [19]:
test_37 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-shorter-baseline-and-trimmed-reference-answers.csv")
test_37.describe()

,repetition,latency,tokens,total_cost,correctness,hallucination
count,20.0,20.000000,20.000000,20.000000,12.000000,13.000000
mean,1.0,17.398945,9455.200000,0.026201,0.791667,0.730769
std,0.0,10.909577,8244.325804,0.022654,0.257464,0.259437
min,1.0,2.687307,0.000000,0.000000,0.500000,0.500000
25%,1.0,4.104023,724.750000,0.000268,0.500000,0.500000
50%,1.0,22.125629,9195.000000,0.027682,1.000000,0.500000
75%,1.0,24.792782,17469.000000,0.047971,1.000000,1.000000
max,1.0,33.988507,21824.000000,0.058036,1.000000,1.000000


In [23]:

print("Test 37: Hallucination")
test_37_hallucination_test = hallucination_score_noramlized(test_37, df_name="test_37")
hallucination_score_mean = test_37["hallucination"].mean() * 2 # denormalize the overall score
print(f"Test 37 Hallucination score: {hallucination_score_mean}")

print(" ")

print("Test 37: Correctnes")
test_37_correctness_test = correctness_score(test_37, df_name="test_37")
correctness_score_mean = test_37["correctness"].mean() * 2 # denormalize the overall score
print(f"Test 37 Correctness score: {correctness_score_mean}")



Test 37: Hallucination
95% CI Hallucination Low - test_37: nan
95% CI Hallucination High - test_37: nan
Standard Error - Hallucination - test_37: nan
hallucination_MOE - test_37: nan
Test 37 Hallucination score: 1.4615384615384615
 
Test 37: Correctnes
95% CI Correctness Low - test_37: nan
95% CI Correctness High - test_37: nan
Standard Error - Correctness - test_37: nan
Correctness MOE - test_37: nan
Test 37 Correctness score: 1.5833333333333333


In [24]:
test_37["hallucination"].isna().sum()

np.int64(7)

# Fitness Form Coach — Optimization Results

## 1. Model Swap: gpt-4o-mini → gpt-5.4-nano

### Nodes Changed
- Router (query classification)
- Chat memory (conversational responses)
- Classifier (exercise classification)

### Latency Comparison

| Query | Before (gpt-4o-mini) | After (gpt-5.4-nano) | Change |
|---|---|---|---|
| "thanks for your help" (chat memory) | 8.21s | 5.32s | -35% |
| "can you tell me again..." (follow-up) | 15.95s | 8.08s | -49% |
| Bench press text query (vector_db → response_gen) | 43.63s | 42.02s | -4% |
| Video query (full pipeline) | 103.80s | 58.11s | -44% |

### Cost Comparison

| Query | Before cost | After cost | Change |
|---|---|---|---|
| "thanks for your help" (chat memory) | $0.0452 | $0.0012 | -97% |
| "can you tell me again..." (follow-up) | $0.0474 | $0.0040 | -92% |
| Bench press text query | $0.0350 | $0.0301 | -14% |
| Video query (full pipeline) | $0.0500 | $0.0452 | -10% |

### Key Takeaways
- Chat memory and follow-up queries saw the biggest improvements: ~35-49% latency reduction and ~90-97% cost reduction.
- Video query latency nearly halved (103.8s → 58.1s), driven by the classifier switch to nano.
- Text queries routed through vector_db → response_generator barely changed (~4%) because the bottleneck is the response_generator LLM — not yet optimized at this stage.

---

## 2. Memory Summarization

### Change
- Removed `RunnableWithMessageHistory` from `response_generator`
- Manually manage history: save summarized response (via gpt-5.4-nano) instead of full video analysis
- User still sees the full response; history stores a ~150-word summary
- Summarization prompt extracts: exercise, main issues, priority fixes, uncertainties, and a concise coaching summary

### Results (same session: video → text query → follow-up → thanks)

| Query | Before (full history) | After (summarized) | Change |
|---|---|---|---|
| "thanks for your help" tokens | 18,300 | 328 | -98% |
| "thanks for your help" latency | 4.52s | 1.74s | -62% |
| Text follow-up tokens | 18,100 | 2,863 | -84% |
| Text follow-up latency | — | 4.48s | — |
| "can you tell me again..." tokens | — | 1,372 | — |

### Key Takeaways
- Token usage on follow-up messages dropped 84-98% because history no longer contains the full video analysis.
- Response quality on follow-ups is unchanged — the summary retains enough context for the model to answer questions about tempo, grip, form cues, etc.
- The summarization adds a one-time nano LLM call after each video analysis, but every subsequent message in the session benefits from the reduced history size.

---

## 3. Model Swap: GPT-5 → GPT-5.4 (Response Generator)

### Node Changed
- Response generator (video analysis)

### Quality Comparison (LangSmith evals, n=20)

| Metric | GPT-5 (baseline) | GPT-5.4 | Change |
|---|---|---|---|
| Correctness | 0.75 | 0.72 | -0.03 (not significant) |
| Hallucination | 0.80 | 0.75 | +0.05 (not significant) |

### Latency Comparison

| Metric | GPT-5 | GPT-5.4 | Change |
|---|---|---|---|
| P50 latency | 53.71s | 24.17s | -55% |
| P99 latency | 103.13s | 35.93s | -65% |

### Key Takeaways
- Correctness and hallucination scores are effectively unchanged — the -0.03 and +0.05 differences are well within the margin of error for 20 samples.
- P50 latency dropped 55%, P99 dropped 65%. The video analysis pipeline is now more than twice as fast.
- GPT-5.4 is a clear win: same quality, dramatically faster.

---

## Cumulative Impact

| Metric | Original | Final | Total Change |
|---|---|---|---|
| Video analysis latency (P50) | 103.80s | 24.17s | -77% |
| "thanks for your help" tokens | 18,300 | 328 | -98% |
| "thanks for your help" latency | 8.21s | 1.74s | -79% |
| "thanks for your help" cost | $0.0452 | $0.0001 | -99% |
| Follow-up query tokens | 18,100 | 1,372 | -92% |